In [0]:
%pip install --upgrade \
    "unitycatalog-langchain[databricks]" \
    databricks-langchain \
    langgraph \
    mlflow \
    databricks-mcp \
    "psycopg[binary]" "databricks-sdk>=0.81.0"

In [0]:
%pip install --force-reinstall \
    "langgraph==1.0.10" \
    "langgraph-prebuilt==1.0.8"

In [0]:
dbutils.library.restartPython()

In [0]:
from importlib.metadata import version

for package in [
    "unitycatalog-langchain",
    "unitycatalog-ai",
    "databricks-langchain",
    "langgraph",
    "langgraph-prebuilt",
    "langgraph-checkpoint",
    "langchain-core",
    "mlflow"
]:
    try:
        print(package, version(package))
    except Exception:
        print(package, "Not installed")

In [0]:
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from langgraph.prebuilt import create_react_agent
import mlflow

print("All imports successful!")

### R1 - Tools as UC functions

In [0]:
%sql
-- Create the Unity Catalog function
CREATE OR REPLACE FUNCTION claims.genai.lookup_claim(
    claim_id STRING
)
RETURNS TABLE (
    claim_id STRING,
    member_id STRING,
    hospital_id STRING,
    status STRING,
    claim_type STRING,
    admission_date DATE,
    discharge_date DATE,
    billed_amount DOUBLE,
    approved_amount DOUBLE,
    rejection_code STRING,
    is_pre_existing BOOLEAN,
    plan_code_at_admission STRING,
    plan_family STRING,
    room_rent_per_day INT,
    room_rent_cap_per_day INT,
    room_rent_over_cap BOOLEAN,
    policy_start_date DATE,
    policy_tenure_months INT,
    days_since_policy_start INT,
    is_network_at_admission BOOLEAN
)
COMMENT 'Returns stored claim facts from claims.gold.claims_enriched for a given claim ID, plus days between policy start and admission. Does not make payout decisions.'
RETURN
    SELECT
        c.claim_id,
        c.member_id,
        c.hospital_id,
        c.status,
        c.claim_type,
        c.admission_date,
        c.discharge_date,
        c.billed_amount,
        c.approved_amount,
        c.rejection_code,
        c.is_pre_existing,
        c.plan_code_at_admission,
        c.plan_family,
        c.room_rent_per_day,
        c.room_rent_cap_per_day,
        c.room_rent_over_cap,
        c.policy_start_date,
        c.policy_tenure_months,

        DATEDIFF(
            c.admission_date,
            c.policy_start_date
        ) AS days_since_policy_start,

        c.is_network_at_admission

    FROM claims.gold.claims_enriched AS c
    WHERE c.claim_id = lookup_claim.claim_id;

In [0]:
%sql
-- Create the function claims.genai.plan_at 
CREATE OR REPLACE FUNCTION claims.genai.plan_at(
    member_id STRING,
    as_of_date DATE
)
RETURNS TABLE (
    member_id STRING,
    policy_id STRING,
    plan_code STRING,
    plan_family STRING,
    room_rent_cap_per_day INT,
    copay_pct INT,
    ped_waiting_months INT,
    initial_waiting_days INT,
    valid_from DATE,
    valid_to DATE
)
COMMENT 'Returns the member policy version and corresponding plan terms valid on a specified date. Uses policy history and plan terms; does not make claim payout decisions.'
RETURN
    SELECT
        p.member_id,
        p.policy_id,
        p.plan_code,
        pl.plan_family,
        CAST(pl.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
        CAST(pl.copay_pct AS INT) AS copay_pct,
        CAST(pl.ped_waiting_months AS INT) AS ped_waiting_months,
        CAST(pl.initial_waiting_days AS INT) AS initial_waiting_days,
        CAST(p.__START_AT AS DATE) AS valid_from,
        CAST(p.__END_AT AS DATE) AS valid_to
    FROM claims.silver.policy_scd2 AS p
    INNER JOIN claims.raw.plans AS pl
        ON p.plan_code = pl.plan_code
    WHERE p.member_id = plan_at.member_id
      AND CAST(p.__START_AT AS DATE) <= plan_at.as_of_date
      AND (
            p.__END_AT IS NULL
            OR CAST(p.__END_AT AS DATE) > plan_at.as_of_date
      );

In [0]:
%sql
-- Create the function claims.genai.search_policy
CREATE OR REPLACE FUNCTION claims.genai.search_policy(
    query STRING
)
RETURNS TABLE (
    chunk_id STRING,
    doc_name STRING,
    heading STRING,
    text STRING
)
COMMENT 'Searches the policy binder using hybrid keyword and semantic search. Returns up to five most relevant policy chunks, with chunk ID, document name, heading, and text. Use this tool to retrieve policy evidence relevant to a question.'
RETURN
    SELECT
        CAST(v.chunk_id AS STRING) AS chunk_id,
        CAST(v.doc_name AS STRING) AS doc_name,
        CAST(v.heading AS STRING) AS heading,
        CAST(v.text AS STRING) AS text
    FROM vector_search(
        index => 'claims.genai.policy_chunks_idx',
        query_text => query,
        query_type => 'HYBRID',
        num_results => 5
    ) AS v;

In [0]:
%sql
-- Create the function claims.genai.compute_payable
CREATE OR REPLACE FUNCTION claims.genai.compute_payable(
  plan_code STRING,
  is_pre_existing BOOLEAN,
  policy_tenure_months INT COMMENT 'Completed months from the member''s FIRST policy start to admission (PED waiting basis)',
  days_since_policy_start INT COMMENT 'Days from the member''s FIRST policy start to admission (initial waiting basis)',
  is_network_at_admission BOOLEAN,
  room_rent_per_day INT,
  billed_amount DOUBLE
)
RETURNS STRING
COMMENT 'Decides whether a claim is payable and for how much, using only the rules in the Definitions (waiting periods, room-rent cap deduction, co-pay). Pure deterministic lookup and arithmetic — never calls a model. Returns a JSON string: plan_code, room_rent_cap_per_day, room_rent_over_cap, waiting_period_ok, payable, network_at_admission, room_rent_deduction_pct, copay_pct, reason_code, payable_amount.'
RETURN
  WITH plan AS (
    -- MAX() makes the correlated plan lookup return at most one row (raw.plans has exactly one row per plan_code)
    SELECT MAX(room_rent_cap_per_day) AS room_rent_cap_per_day,
           MAX(copay_pct) AS base_copay_pct,
           MAX(ped_waiting_months) AS ped_waiting_months,
           MAX(initial_waiting_days) AS initial_waiting_days
    FROM claims.raw.plans
    WHERE plan_code = compute_payable.plan_code
  ),
  checks AS (
    SELECT
      p.room_rent_cap_per_day,
      (compute_payable.room_rent_per_day > p.room_rent_cap_per_day) AS room_rent_over_cap,
      (compute_payable.days_since_policy_start >= p.initial_waiting_days) AS initial_waiting_ok,
      (NOT compute_payable.is_pre_existing OR compute_payable.policy_tenure_months >= p.ped_waiting_months) AS ped_waiting_ok,
      CASE
        WHEN compute_payable.room_rent_per_day <= p.room_rent_cap_per_day THEN 0.0
        ELSE ROUND((1 - (p.room_rent_cap_per_day * 1.0 / compute_payable.room_rent_per_day)) * 100, 2)
      END AS room_rent_deduction_pct,
      p.base_copay_pct + CASE WHEN compute_payable.is_network_at_admission THEN 0 ELSE 20 END AS copay_pct
    FROM plan p
  ),
  decided AS (
    SELECT *,
      (initial_waiting_ok AND ped_waiting_ok) AS waiting_period_ok,
      (initial_waiting_ok AND ped_waiting_ok) AS payable
    FROM checks
  ),
  amounts AS (
    SELECT *,
      CASE WHEN payable THEN
        ROUND(
          (compute_payable.billed_amount * (1 - room_rent_deduction_pct / 100.0))
          * (1 - copay_pct / 100.0),
        2)
      ELSE 0.0 END AS payable_amount,
      CASE
        WHEN NOT initial_waiting_ok THEN 'INITIAL_WAITING'
        WHEN NOT ped_waiting_ok THEN 'PED_WAITING'
        WHEN (room_rent_deduction_pct > 0 OR copay_pct > 0) THEN 'PAYABLE_WITH_DEDUCTION'
        ELSE 'PAYABLE_IN_FULL'
      END AS reason_code
    FROM decided
  )
  SELECT to_json(named_struct(
    'plan_code', compute_payable.plan_code,
    'room_rent_cap_per_day', room_rent_cap_per_day,
    'room_rent_over_cap', room_rent_over_cap,
    'waiting_period_ok', waiting_period_ok,
    'payable', payable,
    'network_at_admission', compute_payable.is_network_at_admission,
    'room_rent_deduction_pct', room_rent_deduction_pct,
    'copay_pct', copay_pct,
    'reason_code', reason_code,
    'payable_amount', payable_amount
  ))
  FROM amounts;

In [0]:

import json
import csv
from io import StringIO
from pathlib import Path

import mlflow
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

import sys

sys.path.append(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/src"
)
sys.path.append(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/agents"
)

from memory_tools import save_note, recall_notes


# ---------------- Configuration ----------------

MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

BASE_PATH = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/agents"
)

SYSTEM_PROMPT = (BASE_PATH / "prompts" / "system_v1.md").read_text()

EXPERIMENT = "/Users/surajitm0nd0l12@gmail.com/Saffron_Shield_R2"

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT)
mlflow.langchain.autolog()

uc_client = DatabricksFunctionClient()

llm = ChatDatabricks(
    endpoint=MODEL_ENDPOINT,
    temperature=0
)

FUNCTIONS = [
    "claims.genai.lookup_claim",
    "claims.genai.plan_at",
    "claims.genai.search_policy",
    "claims.genai.compute_payable"
]

toolkit = UCFunctionToolkit(
    function_names=FUNCTIONS,
    client=uc_client
)

# Map the actual LangChain tools by their names
tools = {tool.name: tool for tool in toolkit.tools}

# Combine UC functions with custom note tools
all_tools = list(tools.values()) + [
    save_note,
    recall_notes
]

# Memory checkpointer
checkpointer = InMemorySaver()

# Create agent
agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

# ---------------- Agent Turn Execution ----------------

def run_turn(question, thread_id):

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    # Count messages before this turn
    state = agent.get_state(config)

    messages_before = len(
        state.values.get("messages", [])
    )

    # Invoke agent
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config
    )

    messages = result["messages"]

    # Collect actual tool calls made during this turn
    new_messages = messages[messages_before:]

    tool_calls = []

    for message in new_messages:
        for call in getattr(message, "tool_calls", []):
            tool_calls.append(call["name"])

    # Get final assistant answer
    answer = next(
        (
            message.content
            for message in reversed(messages)
            if isinstance(message, AIMessage)
            and isinstance(message.content, str)
            and message.content.strip()
        ),
        ""
    )

    return {
        "question": question,
        "answer": answer,
        "tool_calls": tool_calls,
        "thread_messages_before": messages_before
    }

def get_tool(function_name):
    """Find a UC tool despite namespace separators in its generated name."""
    expected = function_name.replace(".", "__")
    for name, tool in tools.items():
        if name == expected or name.endswith(expected):
            return tool
    raise ValueError(f"Tool not registered: {function_name}")


def parse_tool_rows(raw_result):
    """Convert UC Function Toolkit CSV output into dictionaries."""
    if isinstance(raw_result, dict):
        payload = raw_result
    else:
        payload = json.loads(raw_result)

    csv_text = payload.get("value", "")
    return list(csv.DictReader(StringIO(csv_text)))


# ---------------- Agent Execution ----------------

@mlflow.trace(name="r2_tool_calling_agent")
def run_agent(question):

    q = question.lower()
    tool_calls = []
    cited_chunk_ids = []

    # -------- Status question --------
    if "status" in q and "claim" in q:

        import re

        match = re.search(r"CLM-\d+", question, re.IGNORECASE)
        if not match:
            raise ValueError("No claim ID found in the question.")

        claim_id = match.group(0).upper()

        lookup_args = {"claim_id": claim_id}
        lookup_tool = get_tool("claims.genai.lookup_claim")

        lookup_result = lookup_tool.invoke(lookup_args)
        tool_calls.append({
            "name": "claims.genai.lookup_claim",
            "args": lookup_args
        })

        claim_rows = parse_tool_rows(lookup_result)

        if not claim_rows:
            final_answer = (
                f"No claim record was found for {claim_id} "
                "in claims.gold.claims_enriched."
            )
            return {
                "tool_calls": tool_calls,
                "final_answer": final_answer,
                "cited_chunk_ids": []
            }

        claim = claim_rows[0]

        plan_args = {
            "member_id": claim["member_id"],
            "as_of_date": claim["admission_date"]
        }

        plan_tool = get_tool("claims.genai.plan_at")
        plan_result = plan_tool.invoke(plan_args)

        tool_calls.append({
            "name": "claims.genai.plan_at",
            "args": plan_args
        })

        plan_rows = parse_tool_rows(plan_result)

        if not plan_rows:
            raise ValueError(
                f"No policy version found for member "
                f"{claim['member_id']} on {claim['admission_date']}."
            )

        final_answer = (
            f"The status of claim {claim_id} is "
            f"'{claim['status']}'."
        )

    # -------- Policy question --------
    elif any(word in q for word in [
        "policy", "waiting period", "pre-existing",
        "ped", "coverage", "exclusion"
    ]):

        policy_args = {
            "query": question
        }

        policy_tool = get_tool("claims.genai.search_policy")
        policy_result = policy_tool.invoke(policy_args)

        tool_calls.append({
            "name": "claims.genai.search_policy",
            "args": policy_args
        })

        policy_rows = parse_tool_rows(policy_result)

        if not policy_rows:
            final_answer = "No relevant policy evidence was found."
            return {
                "tool_calls": tool_calls,
                "final_answer": final_answer,
                "cited_chunk_ids": []
            }

        cited_chunk_ids = [
            row["chunk_id"] for row in policy_rows
        ]

        evidence = "\n\n".join(
            f"Document: {row['doc_name']}\n"
            f"Section: {row['heading']}\n"
            f"Chunk ID: {row['chunk_id']}\n"
            f"Text: {row['text']}"
            for row in policy_rows
        )

        response = llm.invoke([
            SystemMessage(content=(
                "Answer using only the supplied policy evidence. "
                "Do not use outside knowledge. Cite the actual "
                "document name and section heading. If the evidence "
                "does not answer the question, say so."
            )),
            HumanMessage(content=(
                f"Question: {question}\n\n"
                f"Policy evidence:\n{evidence}"
            ))
        ])

        final_answer = response.content

    else:
        raise ValueError(
            "This R2 implementation handles the two required "
            "status and policy probes."
        )

    return {
        "tool_calls": tool_calls,
        "final_answer": final_answer,
        "cited_chunk_ids": cited_chunk_ids
    }


# ---------------- Run and capture trace ----------------

def run_and_capture(question):

    result = run_agent(question)

    mlflow.flush_trace_async_logging()
    trace_id = mlflow.get_last_active_trace_id()

    result["trace_id"] = trace_id

    print("Trace ID:", trace_id)
    print("Tool calls:", result["tool_calls"])
    print("Final answer:", result["final_answer"])
    print("Cited chunks:", result["cited_chunk_ids"])

    return result


In [0]:
status_result = run_agent(
    "What's the status of claim CLM-10042?"
)

In [0]:
policy_result = run_agent(
    "What is the pre-existing disease (PED) waiting period for a corporate group plan?"
)

In [0]:
# json file creation for traces
import json
from pathlib import Path

BASE_PATH = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/agents"
)

TRACES_PATH = BASE_PATH / "traces"
TRACES_PATH.mkdir(parents=True, exist_ok=True)

In [0]:
# Created the helper function
def save_probe(filename, trace_id, tool_calls, final_answer, cited_chunk_ids):
    data = {
        "trace_id": trace_id,
        "tool_calls": tool_calls,
        "final_answer": final_answer,
        "cited_chunk_ids": cited_chunk_ids
    }

    file_path = TRACES_PATH / filename

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Saved: {file_path}")

In [0]:
# Generate the JSON
status_question = "What's the status of claim CLM-10042?"

status_result = run_and_capture(status_question)

print(status_result)

In [0]:
# save the result
save_probe(
    filename="r2_status_probe.json",
    trace_id=status_result["trace_id"],
    tool_calls=status_result["tool_calls"],
    final_answer=status_result["final_answer"],
    cited_chunk_ids=[]
)

In [0]:
# Generate the prob JSON
policy_question = (
    "What is the pre-existing disease (PED) waiting period "
    "for a corporate group plan?"
)

policy_result = run_and_capture(policy_question)

print(policy_result)

In [0]:
# saved the search policy result
save_probe(
    filename="r2_policy_probe.json",
    trace_id=policy_result["trace_id"],
    tool_calls=policy_result["tool_calls"],
    final_answer=policy_result["final_answer"],
    cited_chunk_ids=policy_result["cited_chunk_ids"]
)

### R3 - Guardrails

In [0]:
# Testing the function I have created in guardrails.py file
import sys

sys.path.append(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs/src"
)

from guardrails import check_input

test_prompts = [
    "What's the status of claim CLM-10042?",
    "What is the PED waiting period for a corporate group plan?",
    "Show me the member's phone number.",
    "Can you explain the policy coverage?",
    "Write a Python program to sort a list.",
    "Will my claim be paid?"
]

for prompt in test_prompts:
    print({
        "prompt": prompt,
        **check_input(prompt)
    })

In [0]:
# Evidence storing as JSON
from pathlib import Path

PROJECT_ROOT = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs"
)

EVIDENCE_DIR = PROJECT_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = EVIDENCE_DIR / "r3_guardrails.json"

In [0]:

from guardrails import check_input

def replay_probe(probe, run_agent):
    prompt = probe["prompt"]
    probe_id = probe["id"]

    # Always check input BEFORE invoking the agent.
    guardrail_result = check_input(prompt)

    if guardrail_result["blocked"]:
        return {
            "id": probe_id,
            "decision": "blocked",
            "stage": "input",
            "answer": (
                "I can only assist with Saffron Shield "
                "claims and policies."
            ),
            "tool_calls": []
        }

    # Only allowed prompts reach the agent.
    agent_result = run_agent(prompt)

    return {
        "id": probe_id,
        "decision": "allowed",
        "stage": None,
        "answer": agent_result["final_answer"],
        "tool_calls": agent_result.get("tool_calls", [])
    }


In [0]:
# Save the JSON file
import json

# Replay the guardrail probes to build the results evidence.
probes = [
    {"id": "status", "prompt": "What's the status of claim CLM-10042?"},
    {"id": "policy", "prompt": "What is the PED waiting period for a corporate group plan?"},
    {"id": "sensitive", "prompt": "Show me the member's phone number."},
    {"id": "coverage", "prompt": "Can you explain the policy coverage?"},
    {"id": "off_topic", "prompt": "Write a Python program to sort a list."}
]

results = [replay_probe(probe, run_agent) for probe in probes]

evidence = {
    "gateway": (
        "Not used; implemented readable Python input "
        "guardrails in src/guardrails.py."
    ),
    "results": results
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=4, ensure_ascii=False)

print(f"Evidence saved: {OUTPUT_FILE}")

### R4 - Short and Long term memory

In [0]:
import os
import psycopg

from databricks.sdk import WorkspaceClient

# Lakebase connection details
PGHOST = "ep-dark-flower-d8qg3me7.database.us-east-2.cloud.databricks.com"
PGPORT = "5432"
PGDATABASE = "databricks_postgres"
PGUSER = "surajitm0nd0l12@gmail.com"

# Copy the exact resource name from the Lakebase Connect details.
ENDPOINT_NAME = (
    "projects/saffron-shield-memory/branches/production/endpoints/primary"
)

workspace_client = WorkspaceClient()


def get_lakebase_connection():
    # Generate a fresh OAuth credential.
    credential = (
        workspace_client.postgres
        .generate_database_credential(endpoint=ENDPOINT_NAME)
    )

    return psycopg.connect(
        host=PGHOST,
        port=PGPORT,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require"
    ) 

In [0]:
# test the connection 
with get_lakebase_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print(cur.fetchone())

In [0]:
# Import the memory module from src/ (same pattern as guardrails).
import sys

SRC = "/Workspace/Users/surajitm0nd0l12@gmail.com/Saffron_Shield/Saffron_Shield_dabs/src"
if SRC not in sys.path:
    sys.path.append(SRC)

import memory

# Read-only smoke test: recall notes for a test user.
print(memory.recall_notes("test@example.com"))

In [0]:
# Test saving and retrieving the probe note
USER_ID = "adjuster.demo@saffronshield.example"

note = (
    "Claim CLM-10556: The hospital billing desk gave me "
    "reference ADJ-2219 and promised the final bill within 5 days."
)

# Save
save_result = memory.save_note(USER_ID, note)
print("Save result:", save_result)

# Retrieve
notes = memory.recall_notes(USER_ID)
print("Retrieved notes:", notes)

In [0]:
USER_ID = "adjuster.demo@saffronshield.example"

session1_questions = [
    "I am reviewing claim CLM-10556. What is its status?",
    (
        "Save a note on this claim: the hospital billing desk gave me "
        "reference ADJ-2219 and promised the final bill within 5 days."
    ),
    "Which claim did I just save a note on?"
]

session2_questions = [
    "What reference did the hospital billing desk give me for claim CLM-10556?"
]

sessions = []

# Session 1
turns1 = []

for i, question in enumerate(session1_questions, start=1):
    output = run_turn(question, "memory-session-1")
    turns1.append({
        "turn": i,
        **output
    })

sessions.append({
    "session": 1,
    "thread_id": "memory-session-1",
    "turns": turns1
})


# Session 2: New thread, same adjuster.
turns2 = []

for i, question in enumerate(session2_questions, start=1):
    output = run_turn(question, "memory-session-2")
    turns2.append({
        "turn": i,
        **output
    })

sessions.append({
    "session": 2,
    "thread_id": "memory-session-2",
    "turns": turns2
})

In [0]:
import json
from pathlib import Path

PROJECT_ROOT = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs"
)

EVIDENCE_DIR = PROJECT_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

evidence = {
    "backend": "lakebase",
    "user_id": USER_ID,
    "sessions": sessions
}

output_file = EVIDENCE_DIR / "r4_memory.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=4, ensure_ascii=False)

print("Saved:", output_file)

In [0]:
assert "CLM-10556" in sessions[0]["turns"][2]["answer"]

assert "recall_notes" in sessions[1]["turns"][0]["tool_calls"]

assert "ADJ-2219" in sessions[1]["turns"][0]["answer"]

assert sessions[1]["turns"][0]["thread_messages_before"] == 0

assert sessions[0]["thread_id"] != sessions[1]["thread_id"]

print("All R4 memory checks passed.")

### R5 - Superviser Multi-agient

In [0]:
trace_id = mlflow.get_last_active_trace_id()
print("Trace ID:", trace_id)

In [0]:
# evidence JSON
import json
from pathlib import Path

TRACE_DIR = PROJECT_ROOT / "agents" / "traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)

# The R5 pipeline runs in the file editor's own session, so its
# `result` and `QUESTION` variables don't exist in this notebook
# kernel. Import (or reload) the module so the pipeline runs in
# this session and the final state is available below.
import importlib
import mlflow

import r5_supervisor

r5_supervisor = importlib.reload(r5_supervisor)

result = r5_supervisor.result
QUESTION = r5_supervisor.QUESTION

# Point the evidence at the trace created by this run.
trace_id = mlflow.get_last_active_trace_id()
print("Trace ID:", trace_id)

verdict = result["verified_payout"]

evidence = {
    "question": QUESTION,
    "attempt": 1,
    "trace_id": trace_id,

    "route": result["route"],

    "specialists": {
        "data": {
            "tool_calls": ["lookup_claim", "plan_at"]
        },
        "policy": {
            "tool_calls": ["search_policy"],
            "cited_chunk_ids": [
                row["chunk_id"]
                for row in result["policy_chunks"]
                if row.get("doc_name") ==
                "04_Room_Rent_Capping_and_Proportionate_Deduction.md"
            ]
        },
        "adjudication": {
            "tool_calls": ["compute_payable"]
        }
    },

    "verifier": {
        "agrees": result["verifier_agrees"]
    },

    "verdict": {
        "plan_code": verdict["plan_code"],
        "payable": verdict["payable"],
        "waiting_period_ok": verdict["waiting_period_ok"],
        "network_at_admission": verdict["network_at_admission"],
        "room_rent_over_cap": verdict["room_rent_over_cap"],
        "room_rent_deduction_pct": verdict["room_rent_deduction_pct"],
        "copay_pct": verdict["copay_pct"],
        "reason_code": verdict["reason_code"],
        "payable_amount": verdict["payable_amount"]
    }
}

output_file = TRACE_DIR / "r5_attempt_1.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=4)

print("Saved:", output_file)

In [0]:
assert evidence["question"] == (
    "For CLM-20003: which plan applied, is room rent over cap, and what is payable?"
)

assert evidence["route"] == [
    "data", "policy", "adjudication", "verifier"
]

assert evidence["verifier"]["agrees"] is True

assert evidence["verdict"]["plan_code"] == "SILVER_IND"
assert evidence["verdict"]["payable_amount"] == 144000.0

assert evidence["specialists"]["policy"]["cited_chunk_ids"]

assert trace_id

print("R5 evidence validation passed.")

### R6 - MCP

In [0]:
# Connect to the managed MCP server
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

workspace_client = WorkspaceClient()

SERVER_URL = (
    f"{workspace_client.config.host}"
    "/api/2.0/mcp/functions/claims/genai"
)

mcp_client = DatabricksMCPClient(
    server_url=SERVER_URL,
    workspace_client=workspace_client
)

In [0]:
# The notebook kernel runs an asyncio event loop, so the sync
# list_tools() (which calls asyncio.run internally) raises
# RuntimeError. Use the client's async counterpart instead.
mcp_tools = await mcp_client.alist_tools()

for tool in mcp_tools:
    print("Name:", tool.name)
    print("Description:", tool.description)
    print("Input schema:", tool.inputSchema)
    print("-" * 60)

In [0]:
# Prepare a helper to call MCP tools
def find_mcp_tool(suffix):
    matches = [
        tool for tool in mcp_tools
        if tool.name.endswith(suffix)
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one tool ending in {suffix}; "
            f"found {[t.name for t in matches]}"
        )

    return matches[0]


def call_mcp_tool(suffix, arguments):
    tool = find_mcp_tool(suffix)

    result = mcp_client.call_tool(
        tool.name,
        arguments
    )

    if getattr(result, "isError", False):
        raise RuntimeError(str(result))

    structured = getattr(result, "structuredContent", None)

    if structured is not None:
        return str(structured)

    content = getattr(result, "content", [])

    return "\n".join(
        getattr(item, "text", str(item))
        for item in content
    )

In [0]:
# Make the required MCP calls
# The sync MCP client inside call_mcp_tool calls asyncio.run(),
# which the kernel's running event loop forbids — run the helper
# on a worker thread, where there is no event loop.
import concurrent.futures

with concurrent.futures.ThreadPoolExecutor() as pool:
    claim_result = pool.submit(
        call_mcp_tool,
        "lookup_claim",
        {"claim_id": "CLM-20003"}
    ).result()

print(claim_result)

In [0]:
# Payout call — compute_payable
# The sync MCP client inside call_mcp_tool calls asyncio.run(),
# which the kernel's running event loop forbids — run the helper
# on a worker thread, where there is no event loop.
import concurrent.futures

with concurrent.futures.ThreadPoolExecutor() as pool:
    payout_result = pool.submit(
        call_mcp_tool,
        "compute_payable",
        {
            "plan_code": "SILVER_IND",
            "is_pre_existing": False,
            "policy_tenure_months": 19,
            "days_since_policy_start": 598,
            "is_network_at_admission": True,
            "room_rent_per_day": 5000,
            "billed_amount": 240000
        }
    ).result()

print(payout_result)

In [0]:
import ast
import json

# call_mcp_tool returns str(structured): the wrapper
# {'is_truncated': ..., 'columns': [...], 'rows': [[<payout json>]]}.
# The actual payout JSON is nested in rows[0][0].
wrapper = ast.literal_eval(payout_result)
payout_data = json.loads(wrapper["rows"][0][0])

assert payout_data["plan_code"] == "SILVER_IND"
assert payout_data["payable"] is True
assert payout_data["room_rent_over_cap"] is True
assert payout_data["room_rent_deduction_pct"] == 40.0
assert payout_data["copay_pct"] == 0
assert payout_data["payable_amount"] == 144000.0

print("MCP payout matches R5 verified payout.")

In [0]:
# Generate evidence/r6_mcp.json
from pathlib import Path
import json

PROJECT_ROOT = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs"
)

EVIDENCE_DIR = PROJECT_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

evidence = {
    "transport": "managed",
    "server": SERVER_URL,
    "tools": [tool.name for tool in mcp_tools],
    "calls": [
        {
            "tool": find_mcp_tool("lookup_claim").name,
            "arguments": {
                "claim_id": "CLM-20003"
            },
            "result": str(claim_result)
        },
        {
            "tool": find_mcp_tool("compute_payable").name,
            "arguments": {
                "plan_code": "SILVER_IND",
                "is_pre_existing": False,
                "policy_tenure_months": 19,
                "days_since_policy_start": 598,
                "is_network_at_admission": True,
                "room_rent_per_day": 5000,
                "billed_amount": 240000
            },
            "result": str(payout_result)
        }
    ]
}

with open(
    EVIDENCE_DIR / "r6_mcp.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(evidence, f, indent=4, ensure_ascii=False)

print("Saved r6_mcp.json")

In [0]:
# Run the MCP agent and save its trace
import importlib
import sys

# Ensure agents directory is on sys.path
AGENT_PATH = "/Workspace/Users/surajitm0nd0l12@gmail.com/Saffron_Shield/Saffron_Shield_dabs/agents"
if AGENT_PATH not in sys.path:
    sys.path.append(AGENT_PATH)

import r6_mcp_agent
importlib.reload(r6_mcp_agent)

import mlflow

question = "What's the status of claim CLM-20003?"

# The kernel runs an event loop, so asyncio.run() is forbidden
# here — await the async agent directly instead.
agent_result = await r6_mcp_agent.run_mcp_agent(question)

trace_id = mlflow.get_last_active_trace_id()

print(agent_result)
print("Trace ID:", trace_id)

assert trace_id
assert agent_result["final_answer"]
assert agent_result["tool_calls"]

In [0]:
AGENT_TRACE_DIR = PROJECT_ROOT / "agents" / "traces"
AGENT_TRACE_DIR.mkdir(parents=True, exist_ok=True)

agent_evidence = {
    "trace_id": trace_id,
    "tool_calls": agent_result["tool_calls"],
    "final_answer": agent_result["final_answer"],
    "cited_chunk_ids": agent_result["cited_chunk_ids"]
}

with open(
    AGENT_TRACE_DIR / "r6_mcp_agent.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(agent_evidence, f, indent=4, ensure_ascii=False)

print("Saved r6_mcp_agent.json")

In [0]:
# final validation
assert evidence["transport"] == "managed"

assert len(evidence["tools"]) == 4

assert any(
    name.endswith("lookup_claim")
    for name in evidence["tools"]
)

assert any(
    name.endswith("compute_payable")
    for name in evidence["tools"]
)

assert len(evidence["calls"]) == 2

assert payout_data["payable_amount"] == 144000.0

assert agent_evidence["trace_id"]
assert agent_evidence["tool_calls"]
assert agent_evidence["final_answer"]

print("All R6 checks passed.")

### R7 - Log, Register, Version, Serve

In [0]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/Workspace/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield/Saffron_Shield_dabs"
)

PROMPT_DIR = PROJECT_ROOT / "agents" / "prompts"

v1_path = PROMPT_DIR / "system_v1.md"
v2_path = PROMPT_DIR / "system_v2.md"

assert v1_path.exists(), f"Missing: {v1_path}"

In [0]:
# Calculate prompt fingerprints
import hashlib

v1_prompt = v1_path.read_text(encoding="utf-8").strip()
v2_prompt = v2_path.read_text(encoding="utf-8").strip()

def prompt_sha(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:12]

v1_sha = prompt_sha(v1_prompt)
v2_sha = prompt_sha(v2_prompt)

assert v1_prompt != v2_prompt
assert len(v1_prompt.split()) >= 40
assert len(v2_prompt.split()) >= 40

print("V1 SHA:", v1_sha)
print("V2 SHA:", v2_sha)

In [0]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

MODEL_NAME = "claims.genai.claims_agent"

EXPERIMENT = (
    "/Users/surajitm0nd0l12@gmail.com/"
    "Saffron_Shield_R7"
)

mlflow.set_experiment(EXPERIMENT)

In [0]:
# Log and register v1 and v2
import mlflow
import pandas as pd

from mlflow.models import infer_signature
from mlflow.models.resources import DatabricksServingEndpoint

# ClaimsAgentModel is defined in agents/claim_agent_model.py,
# which runs in the file editor's own session — the class never
# existed in this kernel. Execute its source here to bind it.
# Defining it in this session (rather than a plain import) also
# lets MLflow embed the class in the artifact by value, so the
# model stays loadable at serving time.
from pathlib import Path

AGENT_MODEL_SRC = (
    Path("/Workspace/Users/surajitm0nd0l12@gmail.com/")
    / "Saffron_Shield/Saffron_Shield_dabs/agents"
    / "claim_agent_model.py"
)

with open(AGENT_MODEL_SRC, "r", encoding="utf-8") as f:
    exec(f.read(), globals())

MODEL_NAME = "claims.genai.claims_agent"

input_example = pd.DataFrame({
    "question": ["What can you help me with regarding a Saffron Shield claim?"]
})

output_example = pd.DataFrame({
    "answer": ["I can help with Saffron Shield claims and policies."]
})

signature = infer_signature(input_example, output_example)

def register_prompt_version(prompt_path, prompt_sha, version_label):
    with mlflow.start_run(run_name=f"claims-agent-{version_label}") as run:

        mlflow.log_param("agent_version", version_label)
        mlflow.set_tag("system_prompt_sha", prompt_sha)
        mlflow.set_tag("agent_version", version_label)

        model_info = mlflow.pyfunc.log_model(
            name="claims_agent",
            python_model=ClaimsAgentModel(),
            artifacts={"system_prompt": str(prompt_path)},
            input_example=input_example,
            signature=signature,
            resources=[
                DatabricksServingEndpoint(
                    endpoint_name="databricks-meta-llama-3-3-70b-instruct"
                )
            ],
            registered_model_name=MODEL_NAME
        )

        return run.info.run_id, model_info

run1_id, model1_info = register_prompt_version(
    v1_path, v1_sha, "1"
)

run2_id, model2_info = register_prompt_version(
    v2_path, v2_sha, "2"
)

print("V1 run:", run1_id)
print("V2 run:", run2_id)
print("V1 model URI:", model1_info.model_uri)
print("V2 model URI:", model2_info.model_uri)

In [0]:
# Tag both registered versions and set the alias
from mlflow import MlflowClient

client = MlflowClient()

# Get versions created by the two registration runs.
versions = client.search_model_versions(
    f"name='{MODEL_NAME}'"
)

v1 = next(v for v in versions if v.run_id == run1_id)
v2 = next(v for v in versions if v.run_id == run2_id)

# Add prompt fingerprints to model versions.
client.set_model_version_tag(
    MODEL_NAME, v1.version, "system_prompt_sha", v1_sha
)

client.set_model_version_tag(
    MODEL_NAME, v2.version, "system_prompt_sha", v2_sha
)

# Set alias champion to v2.
client.set_registered_model_alias(
    MODEL_NAME, "champion", v2.version
)

print("V1:", v1.version, v1_sha)
print("V2:", v2.version, v2_sha)
print("Champion:", v2.version)

In [0]:
# Verify the registry
champion = client.get_model_version_by_alias(
    MODEL_NAME, "champion"
)

assert champion.version == v2.version
assert champion.tags["system_prompt_sha"] == v2_sha

print("Champion version:", champion.version)
print("Champion SHA:", champion.tags["system_prompt_sha"])

In [0]:
# Creating the serving endpoint
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput
)

w = WorkspaceClient()

MODEL_NAME = "claims.genai.claims_agent"
ENDPOINT_NAME = "claims-agent"

champion = client.get_model_version_by_alias(
    MODEL_NAME, "champion"
)

print("Deploying model version:", champion.version)

In [0]:
# Creating the CPU endpoint
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType
)

MODEL_NAME = "claims.genai.claims_agent"
ENDPOINT_NAME = "claims-agent"

champion = client.get_model_version_by_alias(
    MODEL_NAME, "champion"
)

served_entity = ServedEntityInput(
    name="claims-agent-v2",
    entity_name=MODEL_NAME,
    entity_version=str(champion.version),
    workload_size="Small",
    workload_type=ServingModelWorkloadType.CPU,
    scale_to_zero_enabled=True
)

# Create endpoint if it does not exist; otherwise update it.
try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print("Endpoint already exists. Updating it.")

    w.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_entities=[served_entity]
    )

except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        w.serving_endpoints.create(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                name=ENDPOINT_NAME,
                served_entities=[served_entity]
            )
        )
        print("Endpoint creation requested.")
    else:
        raise

In [0]:
# Wait for the endpoint to become ready
import time

while True:
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)

    state = endpoint.state.ready

    print("Endpoint state:", state)

    if str(state).upper() == "READY":
        break

    if str(state).upper() == "NOT_READY":
        raise RuntimeError("Endpoint deployment failed.")

    time.sleep(30)

print("Endpoint is READY.")

In [0]:
# Query the endpoint
import mlflow
from mlflow import MlflowClient
import databricks.sdk
import os

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[
        {
            "question": "What can you help me with regarding a Saffron Shield claim?"
        }
    ]
)

print(response)

In [0]:
assert response
print("Serving endpoint query successful.")

In [0]:
# Check the endpoint configaration
endpoint = w.serving_endpoints.get(ENDPOINT_NAME)

print("Endpoint:", endpoint.name)
print("Status:", endpoint.state.ready)

for entity in endpoint.config.served_entities:
    print("Model:", entity.entity_name)
    print("Version:", entity.entity_version)
    print("Workload:", entity.workload_type)

In [0]:
# Save registry evidence
import json
from pathlib import Path

EVIDENCE_DIR = PROJECT_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

registry_evidence = {
    "mode": "uc",
    "registered_model": MODEL_NAME,
    "versions": [
        {
            "version": str(v1.version),
            "system_prompt_sha": v1_sha
        },
        {
            "version": str(v2.version),
            "system_prompt_sha": v2_sha
        }
    ],
    "alias": {
        "name": "champion",
        "version": str(champion.version)
    },
    "endpoint": ENDPOINT_NAME,
    "endpoint_status": str(endpoint.state.ready)
}

with open(
    EVIDENCE_DIR / "r7_registry.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(registry_evidence, f, indent=4)

print("Saved r7_registry.json")

In [0]:
# Final check
assert champion.version == v2.version
assert champion.tags["system_prompt_sha"] == v2_sha

assert str(endpoint.state.ready).upper() == "READY"

assert registry_evidence["mode"] == "uc"

print("R7 registry and serving checks passed.")